## Agentic ai not diurectly through rag pipleine 

## Through SQL

In [1]:
import sqlite3

In [2]:
connection = sqlite3.connect("mydb.db")


In [3]:
connection

In [4]:
table_creation_query = """
CREATE TABLE IF NOT EXISTS employees (
    emp_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    hire_date TEXT NOT NULL,
    salary REAL NOT NULL
);
"""

In [5]:
table_creation_query2 = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    phone TEXT
);
"""

In [6]:
table_creation_query3 = """
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);
"""

In [7]:
cursor = connection.cursor()

In [8]:
cursor.execute(table_creation_query)
cursor.execute(table_creation_query2)
cursor.execute(table_creation_query3)


In [9]:
insert_query = """
INSERT INTO employees (emp_id, first_name, last_name, email, hire_date, salary)
VALUES (?, ?, ?, ?, ?, ?);
"""

insert_query_customers = """
INSERT INTO customers (customer_id, first_name, last_name, email, phone)
VALUES (?, ?, ?, ?, ?);
"""

insert_query_orders = """
INSERT INTO orders (order_id, customer_id, order_date, amount)
VALUES (?, ?, ?, ?);
"""

In [10]:
employee_data = [
    (1, "Sunny", "Savita", "sunny.sv@abc.com", "2023-06-01", 50000.00),
    (2, "Arhun", "Meheta", "arhun.me@gmail.com", "2022-04-15", 60000.00),
    (3, "Alice", "Johnson", "alice.johnson@jpg.com", "2021-09-30", 55000.00),
    (4, "Bob", "Brown", "bob.brown@uio.com", "2020-01-20", 45000.00),
]

customers_data = [
    (1, "John", "Doe", "john.doe@example.com", "1234567890"),
    (2, "Jane", "Smith", "jane.smith@example.com", "9876543210"),
    (3, "Emily", "Davis", "emily.davis@example.com", "4567891230"),
    (4, "Michael", "Brown", "michael.brown@example.com", "7894561230"),
]

orders_data = [
    (1, 1, "2023-12-01", 250.75),
    (2, 2, "2023-11-20", 150.50),
    (3, 3, "2023-11-25", 300.00),
    (4, 4, "2023-12-02", 450.00),
]

In [11]:
cursor.executemany(insert_query,employee_data)
cursor.executemany(insert_query_customers,customers_data)
cursor.executemany(insert_query_orders,orders_data)

In [12]:
connection.commit()

In [13]:
cursor.execute("select * from orders;")

In [14]:
for row in cursor.fetchall():
    print(row)

(1, 1, '2023-12-01', 250.75)
(2, 2, '2023-11-20', 150.5)
(3, 3, '2023-11-25', 300.0)
(4, 4, '2023-12-02', 450.0)


In [17]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///mydb.db")

In [18]:
db

In [19]:
db.dialect

'sqlite'

# SQL Dialect

## What is a SQL Dialect?

A **SQL dialect** is the specific version or flavor of SQL used by a Database Management System (DBMS).

Although all databases use SQL, each database has its own syntax, functions, and features.

---

## Common SQL Dialects

| Database | SQL Dialect |
|----------|-------------|
| SQLite | SQLite SQL |
| MySQL | MySQL SQL |
| PostgreSQL | PostgreSQL SQL |
| Oracle | Oracle SQL |
| Microsoft SQL Server | T-SQL (Transact-SQL) |

---

## Example

### SQLite

```sql
CREATE TABLE users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT
);
```

### MySQL

```sql
CREATE TABLE users (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100)
);
```

### PostgreSQL

```sql
CREATE TABLE users (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100)
);
```

Although these queries perform the same task, the syntax differs because each database uses a different SQL dialect.

---

# SQL Dialect in LangChain

When creating a database connection:

```python
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///mydb.db")
```

You can check the database dialect using:

```python
print(db.dialect)
```

**Output**

```text
sqlite
```

---

## Why is SQL Dialect Important?

Large Language Models (LLMs) generate SQL queries based on the database dialect.

For example, to retrieve the first 10 records:

| Database | Query |
|----------|-------|
| SQLite | `SELECT * FROM users LIMIT 10;` |
| MySQL | `SELECT * FROM users LIMIT 10;` |
| PostgreSQL | `SELECT * FROM users LIMIT 10;` |
| SQL Server | `SELECT TOP 10 * FROM users;` |
| Oracle | `SELECT * FROM users FETCH FIRST 10 ROWS ONLY;` |

If the wrong dialect is used, the SQL query may fail.

---

## Key Points

- A **SQL dialect** is the database-specific version of SQL.
- Every DBMS has its own syntax and functions.
- LangChain automatically detects the SQL dialect.
- The database dialect helps LLMs generate correct SQL queries.
- Always verify the dialect before executing generated SQL queries.

---

## Check the Current Database Dialect

```python
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///mydb.db")

print("Database Dialect:", db.dialect)
```

**Expected Output**

```text
Database Dialect: sqlite
```

In [20]:
db.get_usable_table_names()

['customers', 'employees', 'orders']

In [22]:
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.1-8b-instant")

In [24]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit


In [25]:
toolkit = SQLDatabaseToolkit(db=db,llm=llm)

In [26]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000024DA12812B0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000024DA12812B0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000024DA12812B0>),
 QuerySQLCheckerTool(description='Use this tool to 

In [28]:
tools = toolkit.get_tools()
for tool in tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [29]:
list_tables_tool = next(
    (tool for tool in tools if tool.name == "sql_db_list_tables"),
    None
)

# Understanding `next()` with Generator Expression

## Code

```python
list_tables_tool = next(
    (tool for tool in tools if tool.name == "sql_db_list_tables"),
    None
)
```

---

# What does this code do?

This line searches through the `tools` list and returns the **first tool** whose name is `"sql_db_list_tables"`.

If no such tool exists, it returns `None`.

---

# Breaking Down the Code

### 1. `tools`

`tools` is a list containing all the tools created by the SQL toolkit.

Example:

```python
tools = [
    Tool(name="sql_db_query"),
    Tool(name="sql_db_schema"),
    Tool(name="sql_db_list_tables"),
    Tool(name="sql_db_query_checker")
]
```

---

### 2. Generator Expression

```python
(tool for tool in tools if tool.name == "sql_db_list_tables")
```

This is called a **generator expression**.

It loops through every tool in the `tools` list.

For each tool, it checks:

```python
tool.name == "sql_db_list_tables"
```

If the condition is `True`, that tool is produced.

It is similar to writing:

```python
for tool in tools:
    if tool.name == "sql_db_list_tables":
        print(tool)
```

The difference is that a generator **does not create a new list**. It produces values one at a time, making it memory efficient.

---

### 3. `next()`

The syntax is:

```python
next(iterator, default_value)
```

- **iterator** → Something that can be iterated over (like a generator).
- **default_value** → Returned if no item is found.

In this example:

```python
next(
    (tool for tool in tools if tool.name == "sql_db_list_tables"),
    None
)
```

means:

> "Return the first tool whose name is `sql_db_list_tables`. If no such tool exists, return `None`."

---

# Equivalent Code Using a `for` Loop

The above code is equivalent to:

```python
list_tables_tool = None

for tool in tools:
    if tool.name == "sql_db_list_tables":
        list_tables_tool = tool
        break
```

### Explanation

- Start with `list_tables_tool = None`.
- Loop through every tool.
- If the tool's name matches `"sql_db_list_tables"`, save it.
- Stop searching using `break`.

---

# Example

Suppose:

```python
tools = [
    Tool(name="sql_db_query"),
    Tool(name="sql_db_schema"),
    Tool(name="sql_db_list_tables"),
    Tool(name="sql_db_query_checker")
]
```

Then,

```python
list_tables_tool = next(
    (tool for tool in tools if tool.name == "sql_db_list_tables"),
    None
)
```

returns:

```python
Tool(name="sql_db_list_tables")
```

---

# What if the tool is not found?

Suppose:

```python
tools = [
    Tool(name="sql_db_query"),
    Tool(name="sql_db_schema")
]
```

Then,

```python
list_tables_tool = next(
    (tool for tool in tools if tool.name == "sql_db_list_tables"),
    None
)
```

returns:

```python
None
```

This prevents an error and lets your program safely check whether the tool exists.

---

# Why Use `next()`?

Using `next()` with a generator expression is:

- ✅ Short and readable
- ✅ Faster because it stops after finding the first match
- ✅ Memory efficient because it doesn't create a new list
- ✅ Common Python practice for retrieving a single matching item

---

# Key Points

- `tools` is a list of available tools.
- The generator expression searches for a tool with a specific name.
- `next()` returns the **first matching tool**.
- If no match is found, `None` is returned.
- This is equivalent to using a `for` loop with `break`, but is more concise and efficient.

In [31]:
list_tables_tool

ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x0000024DA12812B0>)

In [32]:
get_schema_tool = next(
    (tool for tool in tools if tool.name == "sql_db_schema"),
    None
)

In [33]:
list_tables_tool.invoke("")

'customers, employees, orders'

In [34]:
get_schema_tool.invoke("customers")

'\nCREATE TABLE customers (\n\tcustomer_id INTEGER, \n\tfirst_name TEXT NOT NULL, \n\tlast_name TEXT NOT NULL, \n\temail TEXT NOT NULL, \n\tphone TEXT, \n\tPRIMARY KEY (customer_id), \n\tUNIQUE (email)\n)\n\n/*\n3 rows from customers table:\ncustomer_id\tfirst_name\tlast_name\temail\tphone\n1\tJohn\tDoe\tjohn.doe@example.com\t1234567890\n2\tJane\tSmith\tjane.smith@example.com\t9876543210\n3\tEmily\tDavis\temily.davis@example.com\t4567891230\n*/'

In [35]:
print(get_schema_tool.invoke("customers"))


CREATE TABLE customers (
	customer_id INTEGER, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone TEXT, 
	PRIMARY KEY (customer_id), 
	UNIQUE (email)
)

/*
3 rows from customers table:
customer_id	first_name	last_name	email	phone
1	John	Doe	john.doe@example.com	1234567890
2	Jane	Smith	jane.smith@example.com	9876543210
3	Emily	Davis	emily.davis@example.com	4567891230
*/


In [36]:
from langchain_core.tools import tool

@tool
def db_query_tool(query: str) -> str:
    """
    Execute a SQL query against the database and return the result.
    If the query is invalid or returns no result, an error message will be returned.
    In case of an error, the user is advised to rewrite the query and try again.
    """
    result = db.run_no_throw(query)

    if not result:
        return "Error: Query failed. Please rewrite your query and try again."

    return result

In [39]:
db_query_tool.invoke("Select * from Employees;")

"[(1, 'Sunny', 'Savita', 'sunny.sv@abc.com', '2023-06-01', 50000.0), (2, 'Arhun', 'Meheta', 'arhun.me@gmail.com', '2022-04-15', 60000.0), (3, 'Alice', 'Johnson', 'alice.johnson@jpg.com', '2021-09-30', 55000.0), (4, 'Bob', 'Brown', 'bob.brown@uio.com', '2020-01-20', 45000.0)]"

In [40]:
db.run("Select * from Employees;")

"[(1, 'Sunny', 'Savita', 'sunny.sv@abc.com', '2023-06-01', 50000.0), (2, 'Arhun', 'Meheta', 'arhun.me@gmail.com', '2022-04-15', 60000.0), (3, 'Alice', 'Johnson', 'alice.johnson@jpg.com', '2021-09-30', 55000.0), (4, 'Bob', 'Brown', 'bob.brown@uio.com', '2020-01-20', 45000.0)]"

### Difference Between `db.run()` and `db.run_no_throw()`

| `db.run()` | `db.run_no_throw()` |
|------------|---------------------|
| Executes the SQL query. | Executes the SQL query. |
| Raises an exception if the query is invalid. | Does not raise an exception. |
| Stops the program on errors unless handled. | Returns the error as a string and continues execution. |
| Best for debugging. | Best for AI agents and production applications. |

**Example**

```python
# Raises an exception
db.run("SELECT * FORM employees;")
```

```python
# Returns an error message
db.run_no_throw("SELECT * FORM employees;")
```

**Key Point:**  
- `db.run()` → **Throws exceptions** on errors.  
- `db.run_no_throw()` → **Returns error messages** without crashing the program.